**Stage 1.** Plots per-epoch training curves from `training_history.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

# Load the training history data
try:
    history_df = pd.read_csv('/content/training_history.csv')
    print("Training history loaded successfully.")
    print("Available columns:", history_df.columns.tolist())
except FileNotFoundError:
    print("Error: 'training_history.csv' not found. Please ensure the file exists.")
    history_df = pd.DataFrame() # Create an empty DataFrame to avoid errors later

if not history_df.empty:
    # Assuming 'epoch' is the epoch column and the rest are metrics
    # You might need to adjust this based on the actual column names in your CSV
    if 'epoch' in history_df.columns:
        metrics_columns = [col for col in history_df.columns if col != 'epoch']
    else:
        print("Warning: 'epoch' column not found. Assuming the first column is epochs if it looks like epochs, otherwise plotting against index.")
        # Try to infer epoch column or use index
        if history_df.columns[0].lower() == 'epoch' or history_df.columns[0].lower() == 'epochs':
            history_df = history_df.rename(columns={history_df.columns[0]: 'epoch'})
            metrics_columns = [col for col in history_df.columns if col != 'epoch']
        else:
            history_df['epoch'] = history_df.index # Use index as epochs if no epoch column found
            metrics_columns = history_df.columns.tolist()[:-1] # All columns except the newly created epoch

    # Ensure 'epoch' column is integer type
    history_df['epoch'] = history_df['epoch'].astype(int)

    if not metrics_columns:
        print("No metric columns found to plot in 'training_history.csv'.")
    else:
        print("Metrics to plot:", metrics_columns)

        for metric in metrics_columns:
            if pd.api.types.is_numeric_dtype(history_df[metric]):
                plt.figure(figsize=(10, 6)) # Create a new figure for each plot
                sns.lineplot(x='epoch', y=metric, data=history_df, label=metric)
                plt.title(f'{metric} Over Epochs')
                plt.xlabel('Epoch')
                plt.ylabel(metric)
                plt.grid(True)
                plt.legend()
                plt.tight_layout()

                # Force integer ticks on the x-axis
                ax = plt.gca() # Get current axes
                ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

                # Save the figure with an appropriate name
                filename = f'{metric.replace(" ", "_").lower()}_over_20_epochs.png'
                plt.savefig(filename)
                print(f"Saved plot: {filename}")

                plt.show() # Display the plot as well
            else:
                print(f"Skipping non-numeric column '{metric}'.")
else:
    print("Cannot plot: DataFrame is empty.")